# Set Transformer para roles posicionales

Este notebook entrena un modelo a partir del `base_table.csv` etiquetado del dataset común y después lo aplica sobre `data/partidoPrueba/partido_ajustado.mp4` usando el tracking ya generado.

In [ ]:
from pathlib import Path

import pandas as pd

from experiments.positions.set_transformer_pipeline import (
    TrainingConfig,
    predict_roles_for_video,
    train_position_model,
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

BASE_TABLE_PATH = PROJECT_ROOT / "output" / "datasets" / "positions" / "common" / "base_table.csv"
VIDEO_PATH = PROJECT_ROOT / "data" / "partidoPrueba" / "partido_ajustado.mp4"

TRAIN_CONFIG = TrainingConfig(
    epochs=18,
    batch_size=512,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=5,
    seed=42,
)

PROJECT_ROOT, BASE_TABLE_PATH.exists(), VIDEO_PATH.exists()

In [ ]:
training_result = train_position_model(
    project_root=PROJECT_ROOT,
    base_table_path=BASE_TABLE_PATH,
    config=TRAIN_CONFIG,
)

training_result["checkpoint_path"], training_result["metrics"]["val"]["macro_f1"], training_result["metrics"]["test"]["macro_f1"]

In [ ]:
history_df = training_result["history_df"]
history_df.tail()

In [ ]:
prediction_result = predict_roles_for_video(
    model_path=training_result["checkpoint_path"],
    video_path=VIDEO_PATH,
    project_root=PROJECT_ROOT,
)

prediction_result["output_dir"]

In [ ]:
player_summary = pd.read_csv(prediction_result["player_predictions_path"])
frame_predictions = pd.read_csv(prediction_result["frame_predictions_path"])

display(player_summary.sort_values(["team_id", "player_id"]).head(30))
display(frame_predictions[["frame_id", "team_id", "player_id", "class_name", "predicted_role_frame", "predicted_role"]].head(30))